In [31]:
import pandas as pd
import sqlite3

In [32]:
beneficiary_2008 = pd.read_csv('DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv')
inpatient = pd.read_csv('DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv')
outpatient = pd.read_csv('DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv')
beneficiary_2009 = pd.read_csv('DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv')
beneficiary_2010 = pd.read_csv('DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv')

/var/folders/t3/fdm691gd4h301wmr08x1rdz80000gn/T/ipykernel_61784/3964256752.py:3: DtypeWarning: Columns (21,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  outpatient = pd.read_csv('DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv')


In [33]:
conn = sqlite3.connect('medicare.db')

beneficiary_2008.to_sql('beneficiary_2008', conn, if_exists='replace', index=False)
beneficiary_2009.to_sql('beneficiary_2009', conn, if_exists='replace', index=False)
beneficiary_2010.to_sql('beneficiary_2010', conn, if_exists='replace', index=False)
inpatient.to_sql('inpatient', conn, if_exists='replace', index=False)
outpatient.to_sql('outpatient', conn, if_exists='replace', index=False)

print("Done — all 5 tables saved into medicare.db")

Done — all 5 tables saved into medicare.db


In [34]:
# --- Clean inpatient table ---
inpatient['CLM_ADMSN_DT'] = pd.to_datetime(inpatient['CLM_ADMSN_DT'].astype(str), format='%Y%m%d')
inpatient['NCH_BENE_DSCHRG_DT'] = pd.to_datetime(inpatient['NCH_BENE_DSCHRG_DT'].astype(str), format='%Y%m%d')
inpatient = inpatient.drop_duplicates(subset='CLM_ID', keep='first')
inpatient.to_sql('inpatient', conn, if_exists='replace', index=False)
print("Done — inpatient cleaned and saved,", len(inpatient), "rows")

# --- Clean outpatient table ---
outpatient = outpatient.drop_duplicates(subset='CLM_ID', keep='first')
outpatient.to_sql('outpatient', conn, if_exists='replace', index=False)
print("Done — outpatient cleaned and saved,", len(outpatient), "rows")

Done — inpatient cleaned and saved, 66705 rows
Done — outpatient cleaned and saved, 779815 rows


In [35]:
for table in ['beneficiary_2008', 'inpatient', 'outpatient']:
    cols = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 1", conn).columns.tolist()
    print(table, ":", cols)

beneficiary_2008 : ['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR']
inpatient : ['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'CLM_ADMSN_DT', 'ADMTNG_ICD9_DGNS_CD', 'CLM_PASS_THRU_PER_DIEM_AMT', 'NCH_BENE_IP_DDCTBL_AMT', 'NCH_BENE_PTA_COINSRNC_LBLTY_AM', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'CLM_UTLZTN_DAY_CNT', 'NCH_BENE_DSCHRG_DT', 'CLM_DRG_CD', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9

In [52]:
conn.execute("DROP VIEW IF EXISTS all_ids")
conn.execute("""
CREATE VIEW all_ids AS
SELECT DESYNPUF_ID FROM beneficiary_2008
UNION
SELECT DESYNPUF_ID FROM beneficiary_2009
UNION
SELECT DESYNPUF_ID FROM beneficiary_2010;
""")

conn.execute("DROP VIEW IF EXISTS beneficiary_profile")
conn.execute("""
CREATE VIEW beneficiary_profile AS
SELECT
    a.DESYNPUF_ID,
    COALESCE(b10.BENE_BIRTH_DT, b09.BENE_BIRTH_DT, b08.BENE_BIRTH_DT) AS BENE_BIRTH_DT,
    COALESCE(b10.BENE_SEX_IDENT_CD, b09.BENE_SEX_IDENT_CD, b08.BENE_SEX_IDENT_CD) AS BENE_SEX_IDENT_CD,
    COALESCE(b10.BENE_RACE_CD, b09.BENE_RACE_CD, b08.BENE_RACE_CD) AS BENE_RACE_CD,
    COALESCE(b10.SP_STATE_CODE, b09.SP_STATE_CODE, b08.SP_STATE_CODE) AS SP_STATE_CODE,
    COALESCE(b10.BENE_ESRD_IND, b09.BENE_ESRD_IND, b08.BENE_ESRD_IND) AS BENE_ESRD_IND,
    MAX(
        CASE WHEN b08.SP_DIABETES = 1 THEN 1 ELSE 0 END,
        CASE WHEN b09.SP_DIABETES = 1 THEN 1 ELSE 0 END,
        CASE WHEN b10.SP_DIABETES = 1 THEN 1 ELSE 0 END
    ) AS SP_DIABETES,
    MAX(
        CASE WHEN b08.SP_CHF = 1 THEN 1 ELSE 0 END,
        CASE WHEN b09.SP_CHF = 1 THEN 1 ELSE 0 END,
        CASE WHEN b10.SP_CHF = 1 THEN 1 ELSE 0 END
    ) AS SP_CHF,
    MAX(
        CASE WHEN b08.SP_CHRNKIDN = 1 THEN 1 ELSE 0 END,
        CASE WHEN b09.SP_CHRNKIDN = 1 THEN 1 ELSE 0 END,
        CASE WHEN b10.SP_CHRNKIDN = 1 THEN 1 ELSE 0 END
    ) AS SP_CHRNKIDN,
    MAX(
        CASE WHEN b08.SP_ISCHMCHT = 1 THEN 1 ELSE 0 END,
        CASE WHEN b09.SP_ISCHMCHT = 1 THEN 1 ELSE 0 END,
        CASE WHEN b10.SP_ISCHMCHT = 1 THEN 1 ELSE 0 END
    ) AS SP_ISCHMCHT,
    COALESCE(b08.MEDREIMB_IP,0) + COALESCE(b09.MEDREIMB_IP,0) + COALESCE(b10.MEDREIMB_IP,0) AS MEDREIMB_IP_TOTAL,
    COALESCE(b08.MEDREIMB_OP,0) + COALESCE(b09.MEDREIMB_OP,0) + COALESCE(b10.MEDREIMB_OP,0) AS MEDREIMB_OP_TOTAL,
    COALESCE(b08.MEDREIMB_CAR,0) + COALESCE(b09.MEDREIMB_CAR,0) + COALESCE(b10.MEDREIMB_CAR,0) AS MEDREIMB_CAR_TOTAL
FROM all_ids a
LEFT JOIN beneficiary_2008 b08 ON a.DESYNPUF_ID = b08.DESYNPUF_ID
LEFT JOIN beneficiary_2009 b09 ON a.DESYNPUF_ID = b09.DESYNPUF_ID
LEFT JOIN beneficiary_2010 b10 ON a.DESYNPUF_ID = b10.DESYNPUF_ID;
""")
print("Done — views created")

Done — views created


1. Diabetes prevalence + %

In [37]:
sql = """
SELECT COUNT(*) AS total_patients,
       SUM(SP_DIABETES) AS diabetic_patients,
       ROUND(100.0 * SUM(SP_DIABETES) / COUNT(*), 2) AS diabetes_rate_pct
FROM beneficiary_profile;
"""
print(pd.read_sql_query(sql, conn))

   total_patients  diabetic_patients  diabetes_rate_pct
0          116352              64228               55.2


2. Diabetes by sex

In [38]:
sql = """
SELECT BENE_SEX_IDENT_CD, COUNT(*) AS diabetic_count
FROM beneficiary_profile
WHERE SP_DIABETES = 1
GROUP BY BENE_SEX_IDENT_CD;
"""
print(pd.read_sql_query(sql, conn))

   BENE_SEX_IDENT_CD  diabetic_count
0                  1           27436
1                  2           36792


3. Average age: diabetic vs. non-diabetic

In [39]:
sql = """
SELECT SP_DIABETES,
       AVG(2010 - CAST(SUBSTR(BENE_BIRTH_DT, 1, 4) AS INTEGER)) AS avg_age
FROM beneficiary_profile
GROUP BY SP_DIABETES;
"""
print(pd.read_sql_query(sql, conn))

   SP_DIABETES    avg_age
0            0  72.728206
1            1  74.393037


4. Diabetes by race

In [40]:
sql = """
SELECT BENE_RACE_CD, COUNT(*) AS diabetic_count
FROM beneficiary_profile
WHERE SP_DIABETES = 1
GROUP BY BENE_RACE_CD
ORDER BY diabetic_count DESC;
"""
print(pd.read_sql_query(sql, conn))

   BENE_RACE_CD  diabetic_count
0             1           54315
1             2            6267
2             3            2323
3             5            1323


5. Top 10 states by diabetic patient count

In [41]:
sql = """
SELECT SP_STATE_CODE, COUNT(*) AS diabetic_count
FROM beneficiary_profile
WHERE SP_DIABETES = 1
GROUP BY SP_STATE_CODE
ORDER BY diabetic_count DESC
LIMIT 10;
"""
print(pd.read_sql_query(sql, conn))

   SP_STATE_CODE  diabetic_count
0              5            5392
1             10            4508
2             45            4223
3             33            3955
4             39            2867
5             14            2783
6             36            2541
7             23            2393
8             34            2174
9             31            2009


6. Diabetic patients also flagged with CHF, CKD, or ischemic heart disease

In [42]:
sql = """
SELECT
  SUM(SP_CHF) AS diabetes_and_chf,
  SUM(SP_CHRNKIDN) AS diabetes_and_ckd,
  SUM(SP_ISCHMCHT) AS diabetes_and_ischemic_heart
FROM beneficiary_profile
WHERE SP_DIABETES = 1;
"""
print(pd.read_sql_query(sql, conn))

   diabetes_and_chf  diabetes_and_ckd  diabetes_and_ischemic_heart
0             49618             34414                        58392


7. ESRD among diabetics

In [43]:
sql = """
SELECT BENE_ESRD_IND, COUNT(*) AS diabetic_count
FROM beneficiary_profile
WHERE SP_DIABETES = 1
GROUP BY BENE_ESRD_IND;
"""
print(pd.read_sql_query(sql, conn))

  BENE_ESRD_IND  diabetic_count
0             0           56801
1             Y            7427


8. Prevalence rate by year (2008 vs. 2009 vs. 2010)

In [44]:
sql = """
SELECT '2008' AS year, COUNT(*) AS total_patients,
       SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END) AS diabetic_patients,
       ROUND(100.0 * SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS diabetes_rate_pct
FROM beneficiary_2008
UNION ALL
SELECT '2009', COUNT(*), SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END),
       ROUND(100.0 * SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END) / COUNT(*), 2)
FROM beneficiary_2009
UNION ALL
SELECT '2010', COUNT(*), SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END),
       ROUND(100.0 * SUM(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END) / COUNT(*), 2)
FROM beneficiary_2010;
"""
print(pd.read_sql_query(sql, conn))

   year  total_patients  diabetic_patients  diabetes_rate_pct
0  2008          116352              44060              37.87
1  2009          114538              47709              41.65
2  2010          112754              32978              29.25


9. Average total reimbursement (3-year sum): diabetic vs. non-diabetic

In [45]:
sql = """
SELECT SP_DIABETES,
       AVG(MEDREIMB_IP_TOTAL + MEDREIMB_OP_TOTAL + MEDREIMB_CAR_TOTAL) AS avg_total_reimbursement
FROM beneficiary_profile
GROUP BY SP_DIABETES;
"""
print(pd.read_sql_query(sql, conn))

   SP_DIABETES  avg_total_reimbursement
0            0              2456.826414
1            1             17339.916392


10. Average inpatient payment: diabetic vs. non-diabetic

In [46]:
sql = """
SELECT p.SP_DIABETES, AVG(i.CLM_PMT_AMT) AS avg_inpatient_payment
FROM beneficiary_profile p
JOIN inpatient i ON p.DESYNPUF_ID = i.DESYNPUF_ID
GROUP BY p.SP_DIABETES;
"""
print(pd.read_sql_query(sql, conn))

   SP_DIABETES  avg_inpatient_payment
0            0            9826.273784
1            1            9540.943650


11. Average outpatient payment: diabetic vs. non-diabetic

In [47]:
sql = """
SELECT p.SP_DIABETES, AVG(o.CLM_PMT_AMT) AS avg_outpatient_payment
FROM beneficiary_profile p
JOIN outpatient o ON p.DESYNPUF_ID = o.DESYNPUF_ID
GROUP BY p.SP_DIABETES;
"""
print(pd.read_sql_query(sql, conn))

   SP_DIABETES  avg_outpatient_payment
0            0              220.467163
1            1              276.433716


12. Reimbursement scaling with comorbidity count

In [48]:
sql = """
SELECT
  (SP_CHF + SP_CHRNKIDN + SP_ISCHMCHT) AS comorbidity_count,
  AVG(MEDREIMB_IP_TOTAL + MEDREIMB_OP_TOTAL + MEDREIMB_CAR_TOTAL) AS avg_total_reimbursement,
  COUNT(*) AS patient_count
FROM beneficiary_profile
WHERE SP_DIABETES = 1
GROUP BY comorbidity_count
ORDER BY comorbidity_count;
"""
print(pd.read_sql_query(sql, conn))

   comorbidity_count  avg_total_reimbursement  patient_count
0                  0              4043.223294           2696
1                  1              7161.123322          10282
2                  2             11580.819141          21608
3                  3             26278.206936          29642


13. Average length of hospital stay: diabetic vs. non-diabetic

In [49]:
sql = """
SELECT p.SP_DIABETES,
       AVG(julianday(i.NCH_BENE_DSCHRG_DT) - julianday(i.CLM_ADMSN_DT)) AS avg_length_of_stay
FROM beneficiary_profile p
JOIN inpatient i ON p.DESYNPUF_ID = i.DESYNPUF_ID
GROUP BY p.SP_DIABETES;
"""
print(pd.read_sql_query(sql, conn))

   SP_DIABETES  avg_length_of_stay
0            0            5.300035
1            1            5.745482


14. Diabetic patients with both an inpatient and an outpatient claim

In [50]:
sql = """
SELECT COUNT(DISTINCT p.DESYNPUF_ID) AS diabetic_patients_with_both
FROM beneficiary_profile p
JOIN inpatient i ON p.DESYNPUF_ID = i.DESYNPUF_ID
JOIN outpatient o ON p.DESYNPUF_ID = o.DESYNPUF_ID
WHERE p.SP_DIABETES = 1;
"""
print(pd.read_sql_query(sql, conn))

   diabetic_patients_with_both
0                        32140


15. Top 10 diagnosis codes on inpatient claims for diabetic patients

In [51]:
sql = """
SELECT i.ICD9_DGNS_CD_1, COUNT(*) AS claim_count
FROM beneficiary_profile p
JOIN inpatient i ON p.DESYNPUF_ID = i.DESYNPUF_ID
WHERE p.SP_DIABETES = 1 AND i.ICD9_DGNS_CD_1 IS NOT NULL
GROUP BY i.ICD9_DGNS_CD_1
ORDER BY claim_count DESC
LIMIT 10;
"""
print(pd.read_sql_query(sql, conn))

  ICD9_DGNS_CD_1  claim_count
0            486         2265
1          V5789         1660
2          41401         1537
3           0389         1508
4          49121         1461
5           4280         1336
6           5990         1269
7          42731         1123
8          41071         1067
9           5849         1038
